# Check the spread of the null PCA coordinate $s_5$

This notebook tests whether the fifth principal component of the transformed BvK force-constant space is a true physical direction or a numerical/null direction.

The PCA feature vector is

\[
(\alpha_0,\; \alpha_1+2\beta_1,\; \alpha_1-\beta_1,\; \alpha_2,\; \beta_2).
\]

The notebook:

1. Loads `dataframe000013.pkl` through `dataframe000016.pkl`.
2. Selects the top `TOP_K` solutions per `(dataset, mass, a)` ranked by `fitness_norm`.
3. Recomputes PCA on the transformed force constants.
4. Extracts the fifth PCA score,

\[
s_5 = \mathbf{x}_{\rm scaled}\cdot \mathbf{w}_5,
\]

and reports its spread.

If the spread of $s_5$ is near machine precision, PC5 is a numerical null direction. If it has a finite width, it may indicate a hidden constraint or weak but real degree of freedom in the selected high-fitness solutions.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

%config InlineBackend.figure_format = "retina"

# -------------------------------------------------------------------
# User settings
# -------------------------------------------------------------------
DATA_DIR = Path("./dataframes/")  # Update if needed.

DATAFRAME_FILES = {
    "dataframe000013": DATA_DIR / "dataframe000013.pkl",
    "dataframe000014": DATA_DIR / "dataframe000014.pkl",
    "dataframe000015": DATA_DIR / "dataframe000015.pkl",
    "dataframe000016": DATA_DIR / "dataframe000016.pkl",
}

TOP_K = 5
RANK_BY = "fitness_norm"

OUTPUT_DIR = Path("pca_pc5_null_direction_check")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVE_FORMATS = ("pdf", "png")
DPI = 300
FIGSIZE = (6.2, 4.6)


In [ ]:
def find_first_existing(candidates, columns):
    for col in candidates:
        if col in columns:
            return col
    return None


def load_dataframe(path, dataset_label):
    if not path.exists():
        raise FileNotFoundError(f"Could not find {path}. Update DATA_DIR or DATAFRAME_FILES.")
    df = pd.read_pickle(path).copy()
    df["dataset"] = dataset_label
    return df


def apply_publication_axes(ax):
    ax.tick_params(
        axis="both", which="both", direction="in", top=True, right=True,
        labelsize=12, length=5, width=1.1,
    )
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.1)


def save_figure(fig, stem, output_dir=OUTPUT_DIR, formats=SAVE_FORMATS):
    saved = []
    for ext in formats:
        outpath = output_dir / f"{stem}.{ext}"
        fig.savefig(outpath, dpi=DPI, bbox_inches="tight")
        saved.append(outpath)
    print("Saved:")
    for path in saved:
        print(f"  {path}")
    return saved


def display_label(col):
    label_map = {
        "alpha0": r"$\alpha_0$",
        "alpha_0": r"$\alpha_0$",
        "alpha1_plus_2beta1": r"$\alpha_1 + 2\beta_1$",
        "alpha1_minus_beta1": r"$\alpha_1 - \beta_1$",
        "alpha2": r"$\alpha_2$",
        "alpha_2": r"$\alpha_2$",
        "beta2": r"$\beta_2$",
        "beta_2": r"$\beta_2$",
    }
    return label_map.get(col, col)


In [ ]:
# -------------------------------------------------------------------
# Load and select top-k solutions
# -------------------------------------------------------------------
frames = []
for label, path in DATAFRAME_FILES.items():
    tmp = load_dataframe(path, label)
    print(f"{label}: {tmp.shape[0]:,} rows, {tmp.shape[1]:,} columns")
    frames.append(tmp)

df_all = pd.concat(frames, ignore_index=True)
print(f"\nAggregated dataframe before filtering: {df_all.shape[0]:,} rows")

mass_col = find_first_existing(["mass", "m"], df_all.columns)
alat_col = find_first_existing(["a_val", "alat", "a_latt", "lattice_parameter"], df_all.columns)
rank_col = find_first_existing([RANK_BY, "fitness_norm", "fnorm"], df_all.columns)

alpha0_col = find_first_existing(["alpha0", "alpha_0"], df_all.columns)
alpha1_col = find_first_existing(["alpha1", "alpha_1"], df_all.columns)
beta1_col  = find_first_existing(["beta1", "beta_1"], df_all.columns)
alpha2_col = find_first_existing(["alpha2", "alpha_2"], df_all.columns)
beta2_col  = find_first_existing(["beta2", "beta_2"], df_all.columns)

required = [mass_col, alat_col, rank_col, alpha0_col, alpha1_col, beta1_col, alpha2_col, beta2_col]
if any(col is None for col in required):
    raise ValueError(
        "Could not detect all required columns. "
        "Check mass, lattice parameter, rank, and force-constant column names."
    )

print("\nDetected columns:")
for name, col in {
    "mass": mass_col,
    "lattice parameter": alat_col,
    "rank": rank_col,
    "alpha0": alpha0_col,
    "alpha1": alpha1_col,
    "beta1": beta1_col,
    "alpha2": alpha2_col,
    "beta2": beta2_col,
}.items():
    print(f"  {name:18s}: {col}")

df_top = (
    df_all.sort_values(rank_col, ascending=False)
    .groupby(["dataset", mass_col, alat_col], group_keys=False)
    .head(TOP_K)
    .copy()
)

numeric_cols = [mass_col, alat_col, rank_col, alpha0_col, alpha1_col, beta1_col, alpha2_col, beta2_col]
for col in numeric_cols:
    df_top[col] = pd.to_numeric(df_top[col], errors="coerce")

df_top = df_top.dropna(subset=numeric_cols).copy()

# Derived first-neighbor coordinates.
df_top["alpha1_plus_2beta1"] = df_top[alpha1_col] + 2.0 * df_top[beta1_col]
df_top["alpha1_minus_beta1"] = df_top[alpha1_col] - df_top[beta1_col]

force_constant_cols = [
    alpha0_col,
    "alpha1_plus_2beta1",
    "alpha1_minus_beta1",
    alpha2_col,
    beta2_col,
]

print(f"\nRows after top-{TOP_K}-overall selection: {df_top.shape[0]:,}")
print("\nPCA features:")
for col in force_constant_cols:
    print(f"  {col:24s} {display_label(col)}")


In [ ]:
# -------------------------------------------------------------------
# Standardize and compute PCA
# -------------------------------------------------------------------
X = df_top[force_constant_cols].to_numpy(dtype=float)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=5)
scores = pca.fit_transform(X_scaled)

for i in range(scores.shape[1]):
    df_top[f"PC{i+1}"] = scores[:, i]

df_top["s5"] = df_top["PC5"]
df_top["abs_s5"] = df_top["s5"].abs()

explained = pca.explained_variance_ratio_
eigenvalues = pca.explained_variance_
singular_values = pca.singular_values_

print("Explained variance ratio, full precision:")
for i, val in enumerate(explained, start=1):
    print(f"PC{i}: {val:.16e}")

print("\nEigenvalues, full precision:")
for i, val in enumerate(eigenvalues, start=1):
    print(f"lambda_{i}: {val:.16e}")

print("\nSingular values, full precision:")
for i, val in enumerate(singular_values, start=1):
    print(f"sigma_{i}: {val:.16e}")

print("\nNumerical matrix rank checks:")
print(f"rank(X_scaled), default tolerance: {np.linalg.matrix_rank(X_scaled)}")
for tol in [1e-8, 1e-10, 1e-12, 1e-14]:
    print(f"rank(X_scaled), tol={tol:.0e}: {np.linalg.matrix_rank(X_scaled, tol=tol)}")

loadings = pd.DataFrame(
    pca.components_.T,
    index=[display_label(c) for c in force_constant_cols],
    columns=[f"PC{i}" for i in range(1, 6)],
)
loadings


In [ ]:
# -------------------------------------------------------------------
# Quantify the spread of s5
# -------------------------------------------------------------------
s5 = df_top["s5"].to_numpy()
abs_s5 = np.abs(s5)

summary = pd.Series({
    "n": len(s5),
    "mean_s5": np.mean(s5),
    "std_s5_ddof0": np.std(s5, ddof=0),
    "std_s5_ddof1": np.std(s5, ddof=1),
    "min_s5": np.min(s5),
    "q01_s5": np.quantile(s5, 0.01),
    "q05_s5": np.quantile(s5, 0.05),
    "median_s5": np.median(s5),
    "q95_s5": np.quantile(s5, 0.95),
    "q99_s5": np.quantile(s5, 0.99),
    "max_s5": np.max(s5),
    "max_abs_s5": np.max(abs_s5),
    "median_abs_s5": np.median(abs_s5),
})

print("s5 spread summary:")
print(summary.to_string(float_format=lambda x: f"{x:.16e}"))

# A relative scale for comparison: the typical spread of PC1-PC4 scores.
pc_std = df_top[["PC1", "PC2", "PC3", "PC4", "PC5"]].std(ddof=0)
print("\nScore standard deviations:")
print(pc_std.to_string(float_format=lambda x: f"{x:.16e}"))

ratio = summary["std_s5_ddof0"] / pc_std[["PC1", "PC2", "PC3", "PC4"]].mean()
print(f"\nstd(s5) / mean[std(PC1..PC4)] = {ratio:.16e}")

summary.to_csv(OUTPUT_DIR / "pc5_s5_spread_summary.csv")
loadings.to_csv(OUTPUT_DIR / "pca_loadings.csv")
df_top.to_csv(OUTPUT_DIR / "pca_scores_with_s5_top5_overall.csv", index=False)
print(f"\nSaved tables to: {OUTPUT_DIR.resolve()}")


## Interpretation guide

Use the following rule of thumb:

- If `std_s5` and `max_abs_s5` are near machine precision, for example $10^{-14}$--$10^{-12}$ in standardized PCA-score units, PC5 is a numerical null direction.
- If `std_s5` is small but finite, for example $10^{-5}$--$10^{-3}$, the fifth direction may reflect a weak residual degree of freedom or an approximate constraint.
- If `std_s5` is comparable to PC1--PC4 score spreads, PC5 is not actually null.

The most useful quantities are the full-precision eigenvalue of PC5, the singular value of PC5, and the histogram width of $s_5$.


In [ ]:
# -------------------------------------------------------------------
# Histogram of s5
# -------------------------------------------------------------------
fig, ax = plt.subplots(figsize=FIGSIZE)
ax.hist(s5, bins=40, edgecolor="black", linewidth=0.8, alpha=0.8)
ax.axvline(0.0, color="black", linewidth=1.0)
ax.set_xlabel(r"$s_5$", fontsize=15)
ax.set_ylabel("Count", fontsize=15)
ax.set_title(r"Spread of the fifth PCA score $s_5$", fontsize=16, pad=10)
apply_publication_axes(ax)
ax.ticklabel_format(axis="x", style="sci", scilimits=(-3, 3))
fig.tight_layout()
save_figure(fig, "pc5_s5_histogram")
plt.show()


In [ ]:
# -------------------------------------------------------------------
# Absolute s5 values on a log scale
# -------------------------------------------------------------------
fig, ax = plt.subplots(figsize=FIGSIZE)
rank = np.arange(1, len(abs_s5) + 1)
ax.semilogy(rank, np.sort(abs_s5), marker="o", linestyle="none", markersize=4, alpha=0.75)
ax.set_xlabel("Sorted sample index", fontsize=15)
ax.set_ylabel(r"$|s_5|$", fontsize=15)
ax.set_title(r"Sorted absolute fifth PCA score", fontsize=16, pad=10)
apply_publication_axes(ax)
fig.tight_layout()
save_figure(fig, "pc5_abs_s5_sorted_log")
plt.show()


In [ ]:
# -------------------------------------------------------------------
# Check whether s5 varies systematically with family or dataset
# -------------------------------------------------------------------
df_top["abs_alpha2"] = df_top[alpha2_col].abs()
df_top["abs_beta2"] = df_top[beta2_col].abs()
df_top["r_alpha2"] = df_top["abs_alpha2"] / (df_top["abs_alpha2"] + df_top["abs_beta2"] + 1e-12)

fig, ax = plt.subplots(figsize=FIGSIZE)
sc = ax.scatter(
    df_top["r_alpha2"], df_top["s5"],
    c=df_top[rank_col], s=42, alpha=0.8,
    edgecolors="black", linewidths=0.3,
)
ax.axhline(0.0, color="black", linewidth=1.0)
ax.set_xlabel(r"$r_{\alpha_2}=|\alpha_2|/(|\alpha_2|+|\beta_2|)$", fontsize=15)
ax.set_ylabel(r"$s_5$", fontsize=15)
ax.set_title(r"Fifth PCA score versus second-neighbor family coordinate", fontsize=15, pad=10)
apply_publication_axes(ax)
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label(rank_col, fontsize=13)
cbar.ax.tick_params(labelsize=11)
ax.ticklabel_format(axis="y", style="sci", scilimits=(-3, 3))
fig.tight_layout()
save_figure(fig, "pc5_s5_vs_r_alpha2")
plt.show()


In [ ]:
# -------------------------------------------------------------------
# Optional: print the PC5 linear combination in standardized coordinates
# -------------------------------------------------------------------
pc5_loadings = pd.Series(pca.components_[4], index=force_constant_cols)
print("PC5 loading vector in standardized feature coordinates:")
for col, val in pc5_loadings.items():
    print(f"  {display_label(col):35s}: {val:+.16e}")

print("\nThe corresponding score is:")
print("s5 = sum_j loading_j * z_j, where z_j is the standardized feature.")
